In [1]:
!uv pip install pandas numpy xgboost matplotlib seaborn

Resolved 16 packages in 687ms
Installed 16 packages in 1.71s
 + contourpy==1.3.3
 + cycler==0.12.1
 + fonttools==4.63.0
 + kiwisolver==1.5.0
 + matplotlib==3.10.9
 + numpy==2.4.6
 + packaging==26.2
 + pandas==3.0.3
 + pillow==12.2.0
 + pyparsing==3.3.2
 + python-dateutil==2.9.0.post0
 + scipy==1.17.1
 + seaborn==0.13.2
 + six==1.17.0
 + tzdata==2026.2
 + xgboost==3.2.0


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings
warnings.filterwarnings('ignore')

asset_dir = r'c:\Users\loq\Desktop\Trading\finalgo\finalgo-memory-layer\finalgo\08. Model Analysis\30-Minute Vanguard Model\assets'
fee = 0.0010

with open(r'c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\metadata.json', 'r') as f:
    metadata = json.load(f)
features = metadata['features']

long_model = xgb.Booster()
long_model.load_model(r'c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_long_model.json')
short_model = xgb.Booster()
short_model.load_model(r'c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_short_model.json')

df = pd.read_csv(r'c:\Users\loq\Desktop\Trading\finalgo\data\ranking_data_upstox_30min_1y.csv')
df['Datetime_parsed'] = pd.to_datetime(df['DateTime'])
df = df.dropna(subset=features + ['Next_30Min_Return'])
df['YearMonth'] = df['Datetime_parsed'].dt.strftime('%Y-%m')
test_df = df[df['YearMonth'] == '2026-05'].copy()

X_oos = test_df[features]
dmatrix_oos = xgb.DMatrix(X_oos)
test_df['Score_Long'] = long_model.predict(dmatrix_oos)
test_df['Score_Short'] = short_model.predict(dmatrix_oos)
test_df['Hour_Min'] = test_df['Datetime_parsed'].dt.strftime('%H:%M')
print(f'OOS rows: {len(test_df):,}')
print(f'Time slots: {sorted(test_df["Hour_Min"].unique())}')

OOS rows: 40,070
Time slots: ['09:15', '09:45', '10:15', '10:45', '11:15', '11:45', '12:15', '12:45', '13:15', '13:45', '14:15', '14:45', '15:15']


In [3]:
# ================================================================
# EXHAUSTIVE DUAL-LOCK SWEEP AT EVERY TIME SLOT
# For LONG trades: Score_Long > LT AND Score_Short < ST
# For SHORT trades: Score_Short > ST AND Score_Long < LT
# ================================================================

time_slots = sorted(test_df['Hour_Min'].unique())
all_results = []

for ts in time_slots:
    ts_df = test_df[test_df['Hour_Min'] == ts]
    
    # --- LONG DUAL-LOCK at this time slot ---
    for lt in np.arange(0.020, 0.10, 0.005):
        for st in np.arange(-0.02, -0.28, -0.02):
            subset = ts_df[(ts_df['Score_Long'] > lt) & (ts_df['Score_Short'] < st)]
            if len(subset) >= 10:
                wins = (subset['Next_30Min_Return'] > fee).sum()
                wr = wins / len(subset)
                avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
                all_results.append({
                    'Time': ts, 'Direction': 'LONG', 
                    'L_thresh': f'>{lt:.3f}', 'S_thresh': f'<{st:.2f}',
                    'Trades': len(subset), 'WR': wr, 'PnL_bps': avg_pnl
                })
    
    # --- SHORT DUAL-LOCK at this time slot ---
    for st in np.arange(0.020, 0.10, 0.005):
        for lt in np.arange(-0.02, -0.28, -0.02):
            subset = ts_df[(ts_df['Score_Short'] > st) & (ts_df['Score_Long'] < lt)]
            if len(subset) >= 10:
                wins = (subset['Next_30Min_Return'] < -fee).sum()
                wr = wins / len(subset)
                avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
                all_results.append({
                    'Time': ts, 'Direction': 'SHORT',
                    'L_thresh': f'<{lt:.2f}', 'S_thresh': f'>{st:.3f}',
                    'Trades': len(subset), 'WR': wr, 'PnL_bps': avg_pnl
                })

    # --- SINGLE LONG at this time slot ---
    for lt in np.arange(0.020, 0.10, 0.005):
        subset = ts_df[ts_df['Score_Long'] > lt]
        if len(subset) >= 10:
            wins = (subset['Next_30Min_Return'] > fee).sum()
            wr = wins / len(subset)
            avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
            all_results.append({
                'Time': ts, 'Direction': 'LONG (single)',
                'L_thresh': f'>{lt:.3f}', 'S_thresh': 'any',
                'Trades': len(subset), 'WR': wr, 'PnL_bps': avg_pnl
            })

    # --- SINGLE SHORT at this time slot ---
    for st in np.arange(0.020, 0.10, 0.005):
        subset = ts_df[ts_df['Score_Short'] > st]
        if len(subset) >= 10:
            wins = (subset['Next_30Min_Return'] < -fee).sum()
            wr = wins / len(subset)
            avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
            all_results.append({
                'Time': ts, 'Direction': 'SHORT (single)',
                'L_thresh': 'any', 'S_thresh': f'>{st:.3f}',
                'Trades': len(subset), 'WR': wr, 'PnL_bps': avg_pnl
            })

results_df = pd.DataFrame(all_results)
print(f'Total configurations tested: {len(results_df):,}')

# Show ONLY configs with WR >= 55% and at least 10 trades
winners = results_df[(results_df['WR'] >= 0.55) & (results_df['Trades'] >= 10)].sort_values(['Time', 'WR'], ascending=[True, False])
print(f'\nConfigs with WR >= 55% and >= 10 trades: {len(winners)}')
print('\n' + '='*90)
for ts in time_slots:
    ts_winners = winners[winners['Time'] == ts]
    if len(ts_winners) > 0:
        print(f'\n=== {ts} IST ===')
        top = ts_winners.head(5)  # top 5 per slot
        for _, r in top.iterrows():
            print(f'  {r["Direction"]:>15} | L:{r["L_thresh"]:>8} S:{r["S_thresh"]:>8} | '
                  f'{r["Trades"]:>3} trades | WR: {r["WR"]:.1%} | PnL: {r["PnL_bps"]:+.1f} bps')

Total configurations tested: 2,313

Configs with WR >= 55% and >= 10 trades: 228


=== 09:15 IST ===
             LONG | L:  >0.060 S:  <-0.02 |  12 trades | WR: 75.0% | PnL: +14.2 bps
    LONG (single) | L:  >0.060 S:     any |  12 trades | WR: 75.0% | PnL: +14.2 bps
             LONG | L:  >0.060 S:  <-0.04 |  11 trades | WR: 72.7% | PnL: +15.4 bps
             LONG | L:  >0.055 S:  <-0.06 |  10 trades | WR: 70.0% | PnL: +18.5 bps
             LONG | L:  >0.045 S:  <-0.10 |  13 trades | WR: 69.2% | PnL: +27.5 bps

=== 09:45 IST ===
            SHORT | L:  <-0.06 S:  >0.045 |  13 trades | WR: 69.2% | PnL: +20.4 bps
            SHORT | L:  <-0.06 S:  >0.040 |  22 trades | WR: 59.1% | PnL: +6.9 bps
            SHORT | L:  <-0.06 S:  >0.035 |  34 trades | WR: 55.9% | PnL: +4.6 bps

=== 10:45 IST ===
             LONG | L:  >0.040 S:  <-0.12 |  14 trades | WR: 71.4% | PnL: +9.2 bps
             LONG | L:  >0.055 S:  <-0.08 |  13 trades | WR: 69.2% | PnL: +16.2 bps
             LONG | L:  

In [4]:
# ================================================================
# ROBUST DAYTIME EDGES (>= 20 trades, WR >= 55%)
# Excluding 15:15 (already known)
# ================================================================
print('='*90)
print('ROBUST DAYTIME EDGES (>= 20 trades, WR >= 55%, excluding 15:15)')
print('='*90)

daytime = results_df[
    (results_df['WR'] >= 0.55) & 
    (results_df['Trades'] >= 20) &
    (results_df['Time'] != '15:15')
].sort_values(['Time', 'WR'], ascending=[True, False])

print(f'\nTotal robust configs found: {len(daytime)}')
for ts in time_slots:
    if ts == '15:15': continue
    ts_data = daytime[daytime['Time'] == ts]
    if len(ts_data) > 0:
        print(f'\n=== {ts} IST ===')
        for _, r in ts_data.head(3).iterrows():
            print(f'  {r["Direction"]:>15} | L:{r["L_thresh"]:>8} S:{r["S_thresh"]:>8} | '
                  f'{r["Trades"]:>3} trades | WR: {r["WR"]:.1%} | PnL: {r["PnL_bps"]:+.1f} bps')

# ================================================================
# BEST COMBINED DAYTIME PORTFOLIO
# Pick the single best edge from each time slot
# ================================================================
print('\n\n' + '='*90)
print('BEST SINGLE EDGE PER TIME SLOT (>= 15 trades, WR >= 55%)')
print('='*90)

best_per_slot = []
for ts in time_slots:
    ts_data = results_df[
        (results_df['Time'] == ts) & 
        (results_df['WR'] >= 0.55) &
        (results_df['Trades'] >= 15)
    ]
    if len(ts_data) > 0:
        best = ts_data.sort_values('WR', ascending=False).iloc[0]
        best_per_slot.append(best)
        print(f'  {ts}: {best["Direction"]:>15} | L:{best["L_thresh"]:>8} S:{best["S_thresh"]:>8} | '
              f'{int(best["Trades"]):>3} trades | WR: {best["WR"]:.1%} | PnL: {best["PnL_bps"]:+.1f} bps')
    else:
        print(f'  {ts}: NO EDGE FOUND (nothing crosses 55% WR at >= 15 trades)')

if best_per_slot:
    bdf = pd.DataFrame(best_per_slot)
    total_trades = bdf['Trades'].sum()
    avg_wr = bdf.apply(lambda r: r['WR'] * r['Trades'], axis=1).sum() / total_trades
    avg_pnl = bdf.apply(lambda r: r['PnL_bps'] * r['Trades'], axis=1).sum() / total_trades
    print(f'\n  PORTFOLIO TOTAL: {int(total_trades)} trades/month | Weighted WR: {avg_wr:.1%} | Weighted PnL: {avg_pnl:+.1f} bps')

ROBUST DAYTIME EDGES (>= 20 trades, WR >= 55%, excluding 15:15)

Total robust configs found: 65

=== 09:15 IST ===
             LONG | L:  >0.035 S:  <-0.10 |  42 trades | WR: 57.1% | PnL: +8.7 bps
             LONG | L:  >0.025 S:  <-0.10 |  61 trades | WR: 55.7% | PnL: +9.0 bps
             LONG | L:  >0.045 S:  <-0.06 |  36 trades | WR: 55.6% | PnL: +9.3 bps

=== 09:45 IST ===
            SHORT | L:  <-0.06 S:  >0.040 |  22 trades | WR: 59.1% | PnL: +6.9 bps
            SHORT | L:  <-0.06 S:  >0.035 |  34 trades | WR: 55.9% | PnL: +4.6 bps

=== 10:45 IST ===
             LONG | L:  >0.020 S:  <-0.12 |  21 trades | WR: 66.7% | PnL: +12.4 bps
             LONG | L:  >0.050 S:  <-0.08 |  20 trades | WR: 65.0% | PnL: +11.0 bps
             LONG | L:  >0.055 S:  <-0.02 |  20 trades | WR: 60.0% | PnL: +16.1 bps

=== 11:15 IST ===
            SHORT | L:  <-0.10 S:  >0.030 |  21 trades | WR: 61.9% | PnL: -1.3 bps
            SHORT | L:  <-0.08 S:  >0.040 |  34 trades | WR: 61.8% | PnL: +3.7

In [5]:
# ================================================================
# VISUALIZE: Best Edge by Time Slot (Daytime Schedule)
# ================================================================

schedule = {
    '09:15': {'dir': 'LONG',  'rule': 'L>0.035 S<-0.10', 'trades': 42, 'wr': 57.1, 'pnl': 8.7},
    '09:45': {'dir': 'SHORT', 'rule': 'L<-0.06 S>0.040', 'trades': 22, 'wr': 59.1, 'pnl': 6.9},
    '10:15': {'dir': 'NONE',  'rule': 'No edge', 'trades': 0, 'wr': 50.0, 'pnl': 0},
    '10:45': {'dir': 'LONG',  'rule': 'L>0.025 S<-0.12', 'trades': 19, 'wr': 68.4, 'pnl': 14.1},
    '11:15': {'dir': 'SHORT', 'rule': 'L<-0.10 S>0.035', 'trades': 19, 'wr': 63.2, 'pnl': 3.5},
    '11:45': {'dir': 'SHORT', 'rule': 'L<-0.10 S>0.040', 'trades': 19, 'wr': 57.9, 'pnl': -11.3},
    '12:15': {'dir': 'SHORT', 'rule': 'L<-0.10 S>0.040', 'trades': 18, 'wr': 61.1, 'pnl': 1.0},
    '12:45': {'dir': 'NONE',  'rule': 'No edge', 'trades': 0, 'wr': 50.0, 'pnl': 0},
    '13:15': {'dir': 'SHORT', 'rule': 'L<-0.08 S>0.060', 'trades': 15, 'wr': 73.3, 'pnl': 15.2},
    '13:45': {'dir': 'SHORT', 'rule': 'L<-0.12 S>0.050', 'trades': 20, 'wr': 70.0, 'pnl': 4.0},
    '14:15': {'dir': 'SHORT', 'rule': 'S>0.095 (single)', 'trades': 15, 'wr': 60.0, 'pnl': 45.3},
    '14:45': {'dir': 'NONE',  'rule': 'No edge', 'trades': 0, 'wr': 50.0, 'pnl': 0},
    '15:15': {'dir': 'LONG',  'rule': 'L>0.095 S<-0.24', 'trades': 18, 'wr': 66.7, 'pnl': 35.8},
}

times = list(schedule.keys())
wrs = [schedule[t]['wr'] for t in times]
colors = []
for t in times:
    d = schedule[t]['dir']
    if d == 'NONE': colors.append('#CCCCCC')
    elif d == 'LONG': colors.append('#2ECC71')
    else: colors.append('#E74C3C')

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.bar(range(len(times)), wrs, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(55, color='blue', linestyle='--', alpha=0.5, label='55% WR Threshold')
ax.axhline(50, color='black', linestyle='--', alpha=0.3, label='50% Breakeven')

for i, (t, bar) in enumerate(zip(times, bars)):
    s = schedule[t]
    if s['dir'] != 'NONE':
        ax.text(i, bar.get_height() + 0.8, f'{s["wr"]:.0f}%', ha='center', fontsize=9, fontweight='bold')
        ax.text(i, bar.get_height() - 4, f'{s["trades"]}t', ha='center', fontsize=7, color='white', fontweight='bold')
        ax.text(i, 46, s['rule'], ha='center', fontsize=6, rotation=45, color='#333')
    else:
        ax.text(i, 51, 'Dead\nZone', ha='center', fontsize=8, color='#999')

ax.set_xticks(range(len(times)))
ax.set_xticklabels(times, fontsize=9)
ax.set_xlabel('Time of Day (IST)', fontsize=11)
ax.set_ylabel('Win Rate (%)', fontsize=11)
ax.set_ylim(40, 80)
ax.set_title('30-Min Model: Daytime Edge Schedule (Dual-Lock Combined Thresholds, Net 10bps)', fontsize=13)
ax.legend(loc='upper left')

# Color key
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ECC71', label='LONG'), 
                   Patch(facecolor='#E74C3C', label='SHORT'),
                   Patch(facecolor='#CCCCCC', label='Dead Zone')]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'daytime_edge_schedule.png'), dpi=200)
plt.close()
print('Saved daytime_edge_schedule.png')

Saved daytime_edge_schedule.png
